# 原始 tick → 次日信号：Colab 端到端训练

## Goal

这个 notebook 训练共享 DeepLOB + GRU 双头模型。输入是每个股票日 14:55 前最后 200 个十档盘口 snapshot tick；输出是次日开盘到收盘超额收益分数，以及下跌/中性/上涨概率。

训练数据应先在本地从移动硬盘生成。Drive 只保存约 8 GB 的 float16 年度分片、配置、checkpoint 和结果；原始 Parquet 不上传。

## Setup

### Key assumptions

- `RUN_MODE = "smoke"` 使用 178 个真实股票日验证 Drive → Colab → checkpoint 闭环。
- `RUN_MODE = "full"` 使用五年 raw-200 工作集；完成模型选择前不要反复查看 2025 测试结果。
- `DRIVE_PROJECT_DIR` 中包含本项目当前版本，`DRIVE_DATA_DIR` 包含 `manifest.json` 和 `shards/*.npy`。
- Colab GPU 和运行时长不保证，因此所有正式训练都把可恢复 checkpoint 写回 Drive。

In [ ]:
from pathlib import Path

RUN_MODE = "smoke"  # "smoke" 或 "full"
SEED = 0
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/deeplob-reproduction")
LOCAL_PROJECT_DIR = Path("/content/deeplob-reproduction")

if RUN_MODE == "smoke":
    DRIVE_DATA_DIR = Path("/content/drive/MyDrive/deeplob-data/nextday-raw-smoke")
    DRIVE_RUN_DIR = Path("/content/drive/MyDrive/deeplob-runs/raw-200-smoke-colab")
    LOCAL_DATA_DIR = Path("/content/nextday-raw-smoke")
    DATE_SPLIT = {
        "train_start": "2024-01-02",
        "train_end": "2024-01-05",
        "val_start": "2024-01-08",
        "val_end": "2024-01-10",
        "test_start": "2024-01-11",
        "test_end": "2024-01-15",
    }
    EPOCHS = 2
    BATCH_SIZE = 8
    BENCHMARK_BATCHES = 10
    MIN_SYMBOLS_PER_DAY = 10
    PORTFOLIO_QUANTILE = 0.2
    CHECKPOINT_NAME = "raw-200-smoke-colab"
elif RUN_MODE == "full":
    DRIVE_DATA_DIR = Path("/content/drive/MyDrive/deeplob-data/nextday-raw-200")
    DRIVE_RUN_DIR = Path("/content/drive/MyDrive/deeplob-runs/raw-200")
    LOCAL_DATA_DIR = Path("/content/nextday-raw-200")
    DATE_SPLIT = {
        "train_start": "2021-01-01",
        "train_end": "2023-12-31",
        "val_start": "2024-01-01",
        "val_end": "2024-12-31",
        "test_start": "2025-01-01",
        "test_end": "2025-12-31",
    }
    EPOCHS = 30
    BATCH_SIZE = 32
    BENCHMARK_BATCHES = 100
    MIN_SYMBOLS_PER_DAY = 100
    PORTFOLIO_QUANTILE = 0.1
    CHECKPOINT_NAME = "raw-200-dual-head"
else:
    raise ValueError(f"未知 RUN_MODE: {RUN_MODE!r}")

### 1. Mount Drive and stage the project

In [ ]:
import shutil
import subprocess

from google.colab import drive

drive.mount("/content/drive")

if LOCAL_PROJECT_DIR.exists():
    shutil.rmtree(LOCAL_PROJECT_DIR)
shutil.copytree(
    DRIVE_PROJECT_DIR,
    LOCAL_PROJECT_DIR,
    ignore=shutil.ignore_patterns(".venv", "data", "checkpoints*", "__pycache__"),
)
subprocess.run(["python", "-m", "pip", "install", "-q", "-e", str(LOCAL_PROJECT_DIR)], check=True)

### 2. Verify GPU, disk, and source artifacts

In [ ]:
import json

import torch

assert torch.cuda.is_available(), "当前会话没有 CUDA GPU，请在 Runtime 设置中选择 GPU"
assert (DRIVE_DATA_DIR / "manifest.json").is_file(), "Drive 数据清单不存在"
manifest = json.loads((DRIVE_DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
feature_bytes = sum((DRIVE_DATA_DIR / shard["path"]).stat().st_size for shard in manifest["shards"])
free_bytes = shutil.disk_usage("/content").free
assert free_bytes > feature_bytes * 1.2, "Colab 临时盘不足以复制当前工作集"
print(
    {
        "run_mode": RUN_MODE,
        "gpu": torch.cuda.get_device_name(0),
        "samples": len(manifest["samples"]),
        "data_gib": feature_bytes / 2**30,
        "free_gib": free_bytes / 2**30,
    }
)

## Steps

### 3. Copy the current workset to local ephemeral disk

训练期间不要通过 Drive 挂载点随机读取 NPY；先复制到 `/content`，checkpoint 再写回 Drive。

In [ ]:
if LOCAL_DATA_DIR.exists():
    shutil.rmtree(LOCAL_DATA_DIR)
shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR)
assert (LOCAL_DATA_DIR / "manifest.json").stat().st_size > 0
print(f"Local workset ready: {LOCAL_DATA_DIR}")

### 4. Build a dated split and benchmark throughput

In [ ]:
import time

from torch.utils.data import DataLoader

from deeplob.nextday.dataset import NextDayShardDataset
from deeplob.nextday.model import build_nextday_model
from deeplob.nextday.splits import WalkForwardSplit

date_split = WalkForwardSplit.from_strings(**DATE_SPLIT)
train_data = NextDayShardDataset(
    LOCAL_DATA_DIR / "manifest.json", date_split=date_split, split="train"
)
loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)
model = build_nextday_model(
    chunks_per_sample=train_data.chunks_per_sample,
    chunk_size=train_data.chunk_size,
).cuda()
model.train()
started = time.perf_counter()
seen = 0
for batch_index, (features, labels, targets) in enumerate(loader):
    if batch_index >= BENCHMARK_BATCHES:
        break
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        output = model(features.cuda(non_blocking=True))
        loss = torch.nn.functional.cross_entropy(
            output.logits, labels.cuda(non_blocking=True)
        ) + 0.5 * torch.nn.functional.smooth_l1_loss(output.score, targets.cuda(non_blocking=True))
    loss.backward()
    model.zero_grad(set_to_none=True)
    seen += features.shape[0]
elapsed = time.perf_counter() - started
samples_per_second = seen / elapsed
print(
    {
        "run_mode": RUN_MODE,
        "batches": min(len(loader), BENCHMARK_BATCHES),
        "samples_per_second": samples_per_second,
        "estimated_epoch_minutes": len(train_data) / samples_per_second / 60,
    }
)
del model, loader, train_data
torch.cuda.empty_cache()

### 5. Train or resume the dual-head model

In [ ]:
from deeplob.nextday.train import NextDayConfig, train

DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
training_config = NextDayConfig(
    manifest_path=str(LOCAL_DATA_DIR / "manifest.json"),
    train_start=DATE_SPLIT["train_start"],
    train_end=DATE_SPLIT["train_end"],
    val_start=DATE_SPLIT["val_start"],
    val_end=DATE_SPLIT["val_end"],
    test_start=DATE_SPLIT["test_start"],
    test_end=DATE_SPLIT["test_end"],
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=1e-3,
    weight_decay=1e-4,
    patience=8 if RUN_MODE == "full" else 2,
    seed=SEED,
    num_workers=2,
    device="cuda",
    resume=True,
    checkpoint_dir=str(DRIVE_RUN_DIR),
    checkpoint_name=CHECKPOINT_NAME,
    intraday_embedding_size=64,
    day_hidden_size=64,
    dropout=0.1,
    classification_loss_weight=1.0,
    regression_loss_weight=0.5,
    gradient_accumulation_steps=1,
    amp=True,
    min_symbols_per_day=MIN_SYMBOLS_PER_DAY,
    portfolio_quantile=PORTFOLIO_QUANTILE,
)
result = train(training_config)

## Checks

### 6. Inspect the locked-test artifact

只有在验证期配置固定后才运行到这里。正式汇报至少需要 Rank IC、Macro F1、类别分布、按月稳定性和含成本回测；当前训练结果中的 long-short spread 尚未计入交易成本。

In [ ]:
result_path = Path(result["result_file"])
saved_result = json.loads(result_path.read_text(encoding="utf-8"))
summary = {
    "samples": saved_result["samples"],
    "best_validation_metric": saved_result["best_selection_value"],
    "test_rank_ic": saved_result["test"]["daily_rank_ic_mean"],
    "test_macro_f1": saved_result["test"]["macro_f1"],
    "evaluated_test_dates": saved_result["test"]["evaluated_dates"],
}
summary

## Next Steps

1. 先用 `RUN_MODE = "smoke"` 验证 Drive、GPU、训练、checkpoint 和结果读取。
2. 完整 raw-200 数据上传后，把参数单元改为 `RUN_MODE = "full"`，测量 100 个 batch 吞吐。
3. 固定 200-tick 配置并完成至少三个随机种子。
4. 在相同日期和股票池上比较 500 tick / 100 股票，验证更长序列是否有增量。
5. 增加涨跌停、ST、停牌和实际交易成本约束。
6. 客户演示只展示可复现的 checkpoint、输入契约和样本外指标，不把工程 smoke test 描述成盈利证明。